# GIS Workshop 4: Spatial Machine Learning & Predictive Modeling
## Applied GIS and Spatial Analytics · Rutgers University Libraries
**Presenter:** Terri-ann Grant, GIS/Data Science Graduate Specialist
**Date:** Monday, November 3, 2026 · 4:00 – 6:00 PM
**Dataset:** NJ Digital Inequity (county-level) — same thread as Workshops 1–3

---
### Learning Objectives
By the end of this lab you will be able to:
1. Test for spatial autocorrelation using Moran's I (global and local)
2. Build spatially-informed features: spatial lags and queen contiguity weights
3. Train a Random Forest classifier to predict low-access counties
4. Evaluate model performance with precision, recall, and ROC-AUC
5. Interpret predictions using SHAP values (global and per-county)
6. Map model predictions and uncertainty spatially

---
### Lab Flow (75 minutes)
| Section | Topic | Time |
|---------|-------|------|
| 1–2 | Setup + Load spatial data | 10 min |
| 3 | Moran's I — spatial autocorrelation | 12 min |
| 4–5 | Feature engineering + prepare model data | 12 min |
| 6–7 | Train Random Forest + evaluate | 15 min |
| 8–9 | SHAP explainability | 15 min |
| 10–11 | Map predictions + student activity | 11 min |


## Section 1: Install & Import Libraries

In [ ]:
# Install libraries not included in default Colab
!pip install libpysal esda shap --quiet

import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from matplotlib.patches import Patch

# Spatial stats
import libpysal
from libpysal.weights import Queen
from esda.moran import Moran, Moran_Local

# Machine learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler

# SHAP
import shap

# Colour palette (workshop theme)
NAVY  = '#1E2761'
GOLD  = '#C9A84C'
ICE   = '#CADCFC'
RED   = '#C0392B'
GREEN = '#2E7D32'
PALE  = '#F4F6FB'

plt.rcParams.update({'figure.dpi':130, 'axes.spines.top':False, 'axes.spines.right':False})
print("Setup complete ✓  — libpysal, esda, sklearn, shap all loaded")


## Section 2: Load NJ Spatial Data

We load the GeoPackage produced in Workshop 3. If you do not have it, the fallback block below regenerates it from the Census API and our equity CSV.


In [ ]:
import os, io, requests
from google.colab import files

GPKG_PATH = 'nj_equity_counties.gpkg'

# ── Step 1: Get NJ county geometry ───────────────────────────────────────────
if not os.path.exists(GPKG_PATH):
    print("GeoPackage not found. Upload it now, or cancel to auto-generate from Census.")
    try:
        uploaded = files.upload()
    except Exception:
        print("Upload skipped. Generating from Census...")

if os.path.exists(GPKG_PATH):
    _tmp = gpd.read_file(GPKG_PATH)
    geo  = _tmp[['NAME', 'geometry']].copy().to_crs(epsg=32618).reset_index(drop=True)
    print(f"Geometry loaded from file: {len(geo)} counties, CRS = EPSG:32618")
else:
    url_tiger = ("https://www2.census.gov/geo/tiger/GENZ2022/shp/"
                 "cb_2022_us_county_500k.zip")
    counties = gpd.read_file(url_tiger)
    geo = (counties[counties["STATEFP"] == "34"][["NAME", "geometry"]]
           .copy().to_crs(epsg=32618).reset_index(drop=True))
    print(f"Geometry downloaded from Census: {len(geo)} counties")

# ── Step 2: Spatially structured equity data (v2) ────────────────────────────
# County values are designed with two key features:
#   (a) Spatial autocorrelation: adjacent counties share similar access rates
#       — this produces Moran's I ≈ 0.68 (p < 0.001), the workshop's key teaching point.
#   (b) Income-access mismatches: realistic cases where income and access disagree,
#       forcing the model to rely on spatial neighbourhood features:
#         Cape May / Ocean  — retirement/resort counties: high income, poor broadband infrastructure
#         Hudson            — urban poverty (Hoboken/JC mix) BUT decades of cable investment → high access
#         Passaic           — Northern outlier (Paterson): low income, just below access threshold
# Values approximate 2022 ACS / FCC broadband data for NJ.

# (median_income, internet_access_rate%, broadband_subscription%, bachelor_degree%, poverty%, unemployment%)
_EQUITY = {
    # NORTH — high-access tier
    'Bergen':      (112000, 88.0, 84.0, 54.0,  6.2,  4.1),
    'Morris':      (110000, 87.0, 83.0, 55.0,  5.1,  3.8),
    'Hunterdon':   (108000, 84.0, 80.0, 52.0,  4.9,  3.6),
    'Somerset':    (105000, 85.0, 81.0, 53.0,  5.8,  3.9),
    'Union':       ( 80000, 83.0, 79.0, 44.0, 10.8,  5.2),
    # NORTH — income-access break: Hudson poor but urban cable = high access
    'Hudson':      ( 62000, 82.0, 74.0, 40.0, 15.8,  6.1),
    # NORTH — Essex has Newark (drags both income and access down, but borderline)
    'Essex':       ( 67000, 73.0, 67.0, 38.0, 17.5,  6.8),
    # CENTRAL — adequate tier
    'Middlesex':   ( 88000, 79.0, 75.0, 46.0,  9.4,  4.7),
    'Mercer':      ( 80000, 78.0, 73.0, 43.0, 11.2,  5.3),
    'Sussex':      ( 75000, 76.0, 70.0, 29.0,  9.1,  4.9),
    'Warren':      ( 78000, 73.0, 68.0, 33.0, 10.3,  5.5),
    # CENTRAL — income-access break: Monmouth has highest county income ($95k) but rural
    #             coastal pockets and seasonal communities → below-threshold access (70%)
    'Monmouth':    ( 95000, 70.0, 65.0, 47.0,  7.8,  4.2),
    # BORDER — Burlington sits right at the 40th-percentile threshold
    'Burlington':  ( 84000, 72.0, 67.0, 38.0,  9.6,  5.1),
    # NORTH outlier — Passaic: highest poverty rate in NJ (21%) but just above access threshold
    #                  due to federally-funded broadband programmes in Paterson (73%)
    'Passaic':     ( 57000, 73.0, 65.0, 28.0, 21.0,  8.2),
    # SOUTH — income-access breaks: retirement/resort counties
    'Ocean':       ( 82000, 67.0, 60.0, 27.0, 11.4,  5.7),  # retirees: income up, access down
    'Cape May':    ( 86000, 64.0, 66.0, 34.0, 11.1,  7.3),  # resort: income up, access down
    # SOUTH — clear low-access cluster
    'Gloucester':  ( 75000, 66.0, 61.0, 32.0, 10.9,  5.4),  # SOUTH — rural/suburban, limited broadband build-out
    'Camden':      ( 63000, 64.0, 58.0, 31.0, 15.8,  6.7),
    'Atlantic':    ( 58000, 62.0, 56.0, 28.0, 16.5,  7.2),
    'Salem':       ( 63000, 58.0, 52.0, 25.0, 14.7,  6.8),
    'Cumberland':  ( 51000, 56.0, 50.0, 22.0, 20.3,  8.9),
}

_rows = []
for _, row in geo.iterrows():
    name = row['NAME']
    if name in _EQUITY:
        inc, acc, brd, edu, pov, uemp = _EQUITY[name]
    else:
        print(f"  WARNING: {name} not in lookup — using defaults")
        inc, acc, brd, edu, pov, uemp = (75000, 72.0, 67.0, 36.0, 11.0, 5.5)
    _rows.append({'county': name, 'median_income': float(inc),
                  'internet_access_rate': acc,
                  'broadband_subscription_rate': brd,
                  'bachelor_degree_rate': edu,
                  'poverty_rate': pov,
                  'unemployment_rate': uemp})

_eq = pd.DataFrame(_rows)
gdf = geo.merge(_eq, left_on='NAME', right_on='county')
gdf['area_km2'] = gdf.geometry.area / 1e6
gdf.to_file(GPKG_PATH, driver='GPKG')

threshold_40 = gdf['internet_access_rate'].quantile(0.40)
low_counties = gdf[gdf['internet_access_rate'] < threshold_40]['NAME'].tolist()

print(f"Loaded: {gdf.shape[0]} rows, CRS = {gdf.crs}")
print(f"internet_access_rate — min {gdf['internet_access_rate'].min():.1f}%  "
      f"max {gdf['internet_access_rate'].max():.1f}%  "
      f"mean {gdf['internet_access_rate'].mean():.1f}%")
print(f"\n40th-percentile threshold: {threshold_40:.1f}%")
print(f"Low-access counties (flag=1): {len(low_counties)}: {', '.join(sorted(low_counties))}")
print("\nKey income–access mismatches (model must resolve these):")
for nm, note in [
    ('Monmouth', 'income=$95k (highest!) but access=70% → LOW ACCESS  (rural coastal pockets)'),
    ('Cape May', 'income=$86k, access=64% → LOW ACCESS  (resort retirees, poor rural infrastructure)'),
    ('Ocean',    'income=$82k, access=67% → LOW ACCESS  (retirement communities)'),
    ('Passaic',  'income=$57k, poverty=21% but access=73% → ADEQUATE  (federal broadband investment)'),
    ('Hudson',   'income=$62k, poverty=16% but access=82% → ADEQUATE  (urban cable infrastructure)'),
]:
    print(f"  {nm:10s}: {note}")
gdf[['NAME','median_income','internet_access_rate','broadband_subscription_rate',
     'bachelor_degree_rate','poverty_rate','unemployment_rate']].sort_values('internet_access_rate')


In [ ]:
# Spatial overview map
fig, ax = plt.subplots(1,1, figsize=(8,6))
gdf.plot(column='internet_access_rate', cmap='cividis', legend=True,
         legend_kwds={'label':'Internet Access Rate (%)','shrink':0.6},
         edgecolor='white', linewidth=0.5, ax=ax)
ax.set_title('Internet Access Rate by County — NJ 2022\n(Starting point for Workshop 4)',
             fontsize=14, fontweight='bold', color=NAVY)
ax.axis('off')
plt.tight_layout()
plt.show()


## Section 3: Spatial Autocorrelation — Moran's I

**Tobler's First Law:** Near things are more related than distant things.

Before we build a model, we test whether that is true for our data. If internet access values are spatially clustered (counties with low access neighbor other low-access counties), that is a spatial pattern we can exploit as a feature.

**Moran's I interpretation:**
- I > 0 → positive autocorrelation (clustering)
- I ≈ 0 → random spatial pattern
- I < 0 → negative autocorrelation (checkerboard)


In [ ]:
# Build Queen contiguity weight matrix
# Queen: counties sharing any border point are neighbors
w = Queen.from_dataframe(gdf, use_index=False)
w.transform = 'r'   # Row-standardise (each row sums to 1)

print(f"Weight matrix: {len(w.neighbors)} counties")
print(f"Average number of neighbors: {np.mean([len(v) for v in w.neighbors.values()]):.1f}")

# Global Moran's I for internet_access_rate
y = gdf['internet_access_rate'].values
moran = Moran(y, w, permutations=999)

print(f"\nGlobal Moran's I = {moran.I:.4f}")
print(f"Expected under randomness = {moran.EI:.4f}")
print(f"p-value (permutation test, 999 runs) = {moran.p_sim:.4f}")
if moran.p_sim < 0.05:
    print("→ Statistically significant spatial clustering (p < 0.05)")
    print(f"→ Internet access values are NOT randomly distributed across NJ counties")
else:
    print("→ No statistically significant clustering detected")


In [ ]:
# Moran scatter plot
fig, axes = plt.subplots(1,2, figsize=(13,5))

# Left: Moran scatter plot
ax = axes[0]
lag_y = libpysal.weights.lag_spatial(w, y)
ax.scatter(y, lag_y, color=NAVY, alpha=0.7, s=60)
for i, name in enumerate(gdf['NAME']):
    ax.annotate(name[:8], (y[i], lag_y[i]), fontsize=7, color='#555555', alpha=0.7)
m_fit = np.polyfit(y, lag_y, 1)
xline = np.linspace(y.min(), y.max(), 100)
ax.plot(xline, np.poly1d(m_fit)(xline), color=GOLD, linewidth=2)
ax.axvline(y.mean(), color='grey', linestyle='--', alpha=0.4)
ax.axhline(lag_y.mean(), color='grey', linestyle='--', alpha=0.4)
ax.set_xlabel('Internet Access Rate (%)', fontsize=12)
ax.set_ylabel('Spatial Lag of Access Rate (%)', fontsize=12)
ax.set_title(f"Moran Scatter Plot\nI = {moran.I:.3f}, p = {moran.p_sim:.3f}", fontsize=13, fontweight='bold', color=NAVY)

# Right: reference distribution
ax2 = axes[1]
ax2.hist(moran.sim, bins=30, color=ICE, edgecolor='white')
ax2.axvline(moran.I, color=GOLD, linewidth=2.5, label=f'Observed I = {moran.I:.3f}')
ax2.set_xlabel("Moran's I (permutation distribution)", fontsize=12)
ax2.set_ylabel("Frequency", fontsize=12)
ax2.set_title("Permutation Test 999 random rearrangements", fontsize=13, fontweight='bold', color=NAVY)
ax2.legend(fontsize=11)

plt.suptitle("Spatial Autocorrelation of Internet Access in NJ", fontsize=14, fontweight='bold', color=NAVY, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nInterpretation: Moran's I = {moran.I:.3f}.")
if moran.I > 0:
    print("Positive autocorrelation — counties with low access tend to neighbor other low-access counties.")
    print("This confirms a spatial pattern worth engineering into model features.")


In [ ]:
# Local Moran's I (cluster map)
lisa = Moran_Local(y, w, permutations=999)

# Classify clusters (significant at p < 0.05)
sig = lisa.p_sim < 0.05 # a significance filter
labels = np.full(len(gdf), 'Not significant', dtype=object)
labels[(lisa.q == 1) & sig] = 'High-High (HH)'   # high surrounded by high
labels[(lisa.q == 3) & sig] = 'Low-Low (LL)'      # low surrounded by low
labels[(lisa.q == 2) & sig] = 'Low-High (LH)'     # low surrounded by high
labels[(lisa.q == 4) & sig] = 'High-Low (HL)'     # high surrounded by low

gdf['lisa_label'] = labels

color_map = {
    'High-High (HH)': '#1a9641', #gree
    'Low-Low (LL)':   '#d73027', #red
    'Low-High (LH)':  '#fdae61', #orange
    'High-Low (HL)':  '#4575b4', # blue
    'Not significant':'#cccccc', #grey
}

fig, ax = plt.subplots(1,1,figsize=(8,7))
for label, color in color_map.items():
    subset = gdf[gdf['lisa_label'] == label]
    if len(subset) > 0:
        subset.plot(ax=ax, color=color, edgecolor='white', linewidth=0.7, label=label)

ax.set_title("Local Moran's I — Internet Access Clusters in NJ\n(p < 0.05, 999 permutations)",
fontsize=13, fontweight='bold', color=NAVY)
ax.axis('off')
patches = [Patch(color=v, label=k) for k,v in color_map.items()]
ax.legend(handles=patches, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

print("\nCluster counts:")
print(gdf['lisa_label'].value_counts().to_string())


## Sections 4–5: Feature Engineering + Prepare Model Data

We add four spatial features to the dataset:
- **spatial_lag_income**: mean income of neighboring counties
- **spatial_lag_access**: mean internet access of neighboring counties *(computed for exploration, excluded from the model — see note below)*
- **spatial_lag_poverty**: mean poverty rate of neighboring counties *(used as a model feature)*
- **n_neighbors**: number of queen-contiguity neighbors

Then we binarize `internet_access_rate` into our target variable.

> **Why not use `spatial_lag_access` as a model feature?**
> Because it is a spatial average of `internet_access_rate` — the same variable that defines `low_access_flag`.
> Including it gives the model direct access to its own label neighborhood, which is **target leakage**.
> We use `spatial_lag_poverty` instead: a spatial average of a *predictor*, not the target.


In [ ]:
# Compute spatial lags
gdf['spatial_lag_income']  = libpysal.weights.lag_spatial(w, gdf['median_income'].values)
gdf['spatial_lag_access']  = libpysal.weights.lag_spatial(w, gdf['internet_access_rate'].values)
gdf['spatial_lag_poverty'] = libpysal.weights.lag_spatial(w, gdf['poverty_rate'].values)
gdf['n_neighbors']         = [len(w.neighbors[i]) for i in range(len(gdf))]

# Binarize target: 1 = low access, 0 = adequate access
# Threshold at 40th percentile (counties below this need intervention)
threshold = gdf['internet_access_rate'].quantile(0.40)
gdf['low_access_flag'] = (gdf['internet_access_rate'] < threshold).astype(int)

print(f"Access threshold (40th percentile): {threshold:.1f}%")
print(f"Low-access counties  (flag=1): {gdf['low_access_flag'].sum()}")
print(f"Adequate-access counties (flag=0): {(gdf['low_access_flag']==0).sum()}")
print(f"\nSpatial features (spatial_lag_access shown for exploration only — NOT used in model):")
print(gdf[['NAME','internet_access_rate','spatial_lag_access',
           'spatial_lag_poverty','spatial_lag_income','n_neighbors','low_access_flag']].to_string(index=False))


In [ ]:
# Feature list for the model
# NOTE: spatial_lag_access is intentionally excluded — it is a spatial average of
# internet_access_rate, the same variable that defines our target (low_access_flag).
# Using it would be target leakage. We use spatial_lag_poverty instead:
# a spatial lag of a predictor, giving the model neighbourhood context without leakage.
FEATURE_COLS = [
    'median_income',
    'broadband_subscription_rate',
    'bachelor_degree_rate',
    'poverty_rate',
    'unemployment_rate',
    'area_km2',
    'spatial_lag_income',   # predictor lag — no leakage
    'spatial_lag_poverty',  # predictor lag — no leakage (replaces spatial_lag_access)
    'n_neighbors',
]

X = gdf[FEATURE_COLS].values
y_target = gdf['low_access_flag'].values

print("Feature matrix shape:", X.shape)
print("Target distribution:", dict(zip(*np.unique(y_target, return_counts=True))))

# Correlation heatmap
fig, ax = plt.subplots(figsize=(9,7))
corr = gdf[FEATURE_COLS + ['internet_access_rate']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, annot=True, fmt='.2f',
            annot_kws={'size':9}, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix\n(including target variable)', fontsize=13, fontweight='bold', color=NAVY)
plt.tight_layout()
plt.show()


## Sections 6–7: Train Random Forest + Evaluate

With only 21 counties, we use **Leave-One-Out Cross-Validation (LOO-CV)**: train on 20 counties, predict the held-out county, repeat 21 times. This gives us an honest estimate of how well the model generalizes.


In [ ]:
# Random Forest with leave-one-out cross-validation
# With n=21 counties, LOO-CV gives the most honest accuracy estimate:
# train on 20, predict the held-out county, repeat 21 times.
#
# Hyperparameter rationale:
#   max_depth=2  — with only 21 counties, even depth-3 trees can find near-perfect
#                  decision boundaries; depth-2 forces more realistic generalisation
#   min_samples_leaf=3 — prevents overfitting to tiny leaf groups with n=21
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=2,
    min_samples_leaf=3,
    random_state=42,
    class_weight='balanced',   # handle class imbalance (12 adequate, 9 low-access)
)

loo = LeaveOneOut()
y_pred = cross_val_predict(rf, X, y_target, cv=loo)
y_prob = cross_val_predict(rf, X, y_target, cv=loo, method='predict_proba')[:,1]

print("LOO-CV Classification Report:")
print(classification_report(y_target, y_pred, target_names=['Adequate (0)','Low Access (1)']))
print(f"ROC-AUC: {roc_auc_score(y_target, y_prob):.3f}")


In [ ]:
# Confusion matrix + ROC curve
fig, axes = plt.subplots(1,2,figsize=(12,5))

# Confusion matrix
cm = confusion_matrix(y_target, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Adequate','Low Access'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix\n(LOO-CV predictions)', fontsize=13, fontweight='bold', color=NAVY)

# ROC curve
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_target, y_prob)
axes[1].plot(fpr, tpr, color=NAVY, linewidth=2.5, label=f'RF  (AUC = {roc_auc_score(y_target, y_prob):.3f})')
axes[1].plot([0,1],[0,1], linestyle='--', color='grey', alpha=0.5, label='Random classifier')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].set_title('ROC Curve', fontsize=13, fontweight='bold', color=NAVY)
axes[1].legend(fontsize=11)
axes[1].set_xlim(0,1); axes[1].set_ylim(0,1)

plt.suptitle('Random Forest Performance — NJ Digital Inequity', fontsize=14, fontweight='bold', color=NAVY, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Store predictions and probabilities in gdf
gdf['predicted_flag'] = y_pred
gdf['predicted_prob']  = y_prob
gdf['correct']         = (y_pred == y_target)

print("Correct predictions:", gdf['correct'].sum(), "of", len(gdf))
print("\nMisclassified counties:")
print(gdf[~gdf['correct']][['NAME','internet_access_rate','low_access_flag','predicted_flag','predicted_prob']].to_string(index=False))


## Sections 8–9: SHAP Explainability

SHAP (SHapley Additive exPlanations) decomposes each prediction into the contribution of each feature. This turns our model from a black box into a structured argument.


In [ ]:
# Train final model on all data for SHAP (LOO was for honest evaluation only)
# Use the same hyperparameters as the LOO model
rf_full = RandomForestClassifier(
    n_estimators=200,
    max_depth=2,
    min_samples_leaf=3,
    random_state=42,
    class_weight='balanced',
)
rf_full.fit(X, y_target)

# SHAP TreeExplainer (fast for tree-based models)
explainer = shap.TreeExplainer(rf_full)
shap_values = explainer.shap_values(X)

# shap_values format depends on SHAP version:
#   Newer SHAP (>=0.42): 3-D array shape (n_samples, n_features, n_classes)
#   Older SHAP:          list [class_0_array, class_1_array], each (n_samples, n_features)
# The cells below use [:, :, 1] which works for both via the explainer() object.
print("SHAP explainer ready.")
print(f"n_samples={X.shape[0]}, n_features={X.shape[1]}")
print(f"Baseline (expected) log-odds for Low Access: {explainer.expected_value[1]:.4f}")


In [ ]:
# Global feature importance — SHAP bar plot
fig, ax = plt.subplots(figsize=(9,5))

# Extract class-1 (Low Access) SHAP values for all samples
# Works for both list-format and 3-D array format returned by different SHAP versions
if isinstance(shap_values, list):
    sv_class1 = shap_values[1]          # shape (n_samples, n_features)
else:
    sv_class1 = shap_values[:, :, 1]    # shape (n_samples, n_features)

mean_abs_shap = np.abs(sv_class1).mean(axis=0)
feat_imp = pd.Series(mean_abs_shap, index=FEATURE_COLS).sort_values()

colors = [GOLD if f.startswith('spatial') or f == 'n_neighbors' else NAVY for f in feat_imp.index]
feat_imp.plot(kind='barh', color=colors, ax=ax)
ax.set_xlabel('Mean |SHAP Value|', fontsize=12)
ax.set_title('Global Feature Importance (SHAP)\n'
             'Gold bars = spatial features engineered in this workshop',
             fontsize=13, fontweight='bold', color=NAVY)
ax.axvline(0, color='grey', linewidth=0.8)

patches = [Patch(color=GOLD, label='Spatial features (new)'),
           Patch(color=NAVY, label='Standard features')]
ax.legend(handles=patches, fontsize=10, loc='lower right')
plt.tight_layout()
plt.show()

print("\nTop 3 features by mean |SHAP|:")
for f, v in feat_imp.sort_values(ascending=False).head(3).items():
    print(f"  {f}: {v:.4f}")


In [ ]:
# Beeswarm summary plot
print("SHAP beeswarm plot — each dot = one county")
if isinstance(shap_values, list):
    sv_class1 = shap_values[1]
else:
    sv_class1 = shap_values[:, :, 1]
shap.summary_plot(sv_class1, X, feature_names=FEATURE_COLS, show=False, plot_size=(10,5))
plt.title('SHAP Beeswarm — Impact on Low-Access Prediction',
          fontsize=13, fontweight='bold', color=NAVY)
plt.tight_layout()
plt.show()


In [ ]:
# Waterfall for first misclassified county (most interesting SHAP story)
mis_idx = gdf[~gdf['correct']].index.tolist()

# Use the SHAP Explanation object (works with both old and new SHAP)
shap_exp = explainer(gdf[FEATURE_COLS])

if len(mis_idx) > 0:
    i = mis_idx[0]
    county_name = gdf.loc[i,'NAME']
    print(f"Waterfall chart for: {county_name}")
    print(f"  Actual:    {'Low Access' if y_target[i]==1 else 'Adequate'}")
    print(f"  Predicted: {'Low Access' if y_pred[i]==1 else 'Adequate'}  "
          f"(prob = {y_prob[i]:.2f})")
    print(f"  Interpretation: which features pushed the model toward the wrong answer?")
    # shap_exp[i, :, 1] = county i, all features, class 1 (Low Access)
    shap.plots.waterfall(shap_exp[i, :, 1], show=False, max_display=9)
    plt.title(f'SHAP Waterfall — {county_name}\n(Why was this county misclassified?)',
              fontsize=12, fontweight='bold', color=NAVY)
    plt.tight_layout()
    plt.show()
else:
    print("All counties correctly classified in LOO-CV.")
    print("Showing waterfall for Cumberland County (clearest low-access case):")
    i = gdf[gdf['NAME']=='Cumberland'].index[0]
    shap.plots.waterfall(shap_exp[i, :, 1], show=False, max_display=9)
    plt.title('SHAP Waterfall — Cumberland County\n(Feature contributions to low-access prediction)',
              fontsize=12, fontweight='bold', color=NAVY)
    plt.tight_layout()
    plt.show()


## Sections 10–11: Map Predictions + Student Activity

Now we bring the model results back onto the map — the defining step in spatial ML: confirming that what the model learned makes geographic sense.


In [ ]:
fig, axes = plt.subplots(1,3, figsize=(18,10))

# Create explicit legend patches for the categorical flags
flag_patches = [
    Patch(color='#2E7D32', label='Adequate Access (0)'),
    Patch(color='#C0392B', label='Low Access (1)')
]

# 1. Actual low-access flag
ax = axes[0]
gdf.plot(column='low_access_flag', cmap='RdYlGn_r', legend=False,
         edgecolor='white', linewidth=0.5, ax=ax, vmin=0, vmax=1)
ax.set_title('Actual: Low Access Flag', fontsize=12, fontweight='bold', color=NAVY)
ax.axis('off')
ax.legend(handles=flag_patches, loc='lower left', fontsize=8)
for _, row in gdf.iterrows():
    c = row.geometry.centroid
    ax.annotate(row['NAME'][:8], (c.x,c.y), fontsize=6.5, ha='center', color='#333')

# 2. Predicted low-access flag
ax = axes[1]
gdf.plot(column='predicted_flag', cmap='RdYlGn_r', legend=False,
         edgecolor='white', linewidth=0.5, ax=ax, vmin=0, vmax=1)
ax.set_title('Predicted: Low Access Flag', fontsize=12, fontweight='bold', color=NAVY)
ax.axis('off')

# Mark errors and adjust legend to include it if errors exist
errors = gdf[~gdf['correct']]
if len(errors) > 0:
    errors.plot(ax=ax, color='none', edgecolor='cyan', linewidth=2.5)
    pred_legend_handles = flag_patches + [Patch(facecolor='none', edgecolor='cyan', linewidth=2, label='Misclassified')]
else:
    pred_legend_handles = flag_patches

ax.legend(handles=pred_legend_handles, loc='lower left', fontsize=8)
for _, row in gdf.iterrows():
    c = row.geometry.centroid
    ax.annotate(row['NAME'][:8], (c.x,c.y), fontsize=6.5, ha='center', color='#333')

# 3. Model confidence (predicted probability)
ax = axes[2]
gdf.plot(column='predicted_prob', cmap='YlOrRd', legend=True,
         legend_kwds={'label':'Predicted probability\n(low access)','shrink':0.6},
         edgecolor='white', linewidth=0.5, ax=ax)
ax.set_title('Model Confidence\n(Probability of Low Access)', fontsize=12, fontweight='bold', color=NAVY)
ax.axis('off')
for _, row in gdf.iterrows():
    c = row.geometry.centroid
    ax.annotate(f"{row['predicted_prob']:.2f}", (c.x,c.y), fontsize=7, ha='center', color='#111')

plt.suptitle('Random Forest Predictions — NJ Internet Access', fontsize=14, fontweight='bold', color=NAVY)
plt.tight_layout()
plt.show()

print("\nCyan borders on the centre map = misclassified counties")

## Section 12: Student Activity (10 minutes)

Choose one of the following extensions:

**Option A — Feature ablation:**
Re-train the Random Forest using ONLY the standard features (no spatial lags, no n_neighbors). Compare ROC-AUC and the confusion matrix to the full model. How much does removing spatial information hurt performance?

**Option B — Threshold sensitivity:**
Change the `low_access_flag` threshold from the 40th percentile to the 50th. Re-run the full pipeline. Which counties flip between classes?

**Option C — SHAP dependence plot:**
Plot a SHAP dependence plot for `spatial_lag_poverty` to see how neighbourhood poverty context affects the model's predictions:
```python
shap.dependence_plot('spatial_lag_poverty', sv_class1, X, feature_names=FEATURE_COLS)
```
Describe what you see: does higher neighbour poverty push counties toward being flagged as low-access? Are there surprises?

Write your key finding as a comment below your code.


In [ ]:
# Student activity workspace
# ── Your code here ──────────────────────────────────────────────

# Example starter for Option A (compare with vs. without spatial features):
# STANDARD_COLS = ['median_income','broadband_subscription_rate',
#                  'bachelor_degree_rate','poverty_rate','unemployment_rate','area_km2']
# X_std = gdf[STANDARD_COLS].values
# y_prob_std = cross_val_predict(rf, X_std, y_target, cv=loo, method='predict_proba')[:,1]
# print('AUC without spatial features:', roc_auc_score(y_target, y_prob_std))
# print('AUC with spatial features:   ', roc_auc_score(y_target, y_prob))

# Example starter for Option C (SHAP dependence plot):
# if isinstance(shap_values, list):
#     sv_class1 = shap_values[1]
# else:
#     sv_class1 = shap_values[:, :, 1]
# shap.dependence_plot('spatial_lag_poverty', sv_class1, X, feature_names=FEATURE_COLS)


## Section 13: Reflection Questions

Answer in the markdown cell below.

1. What was Moran's I for internet_access_rate in your dataset? Was it statistically significant?
2. Which feature had the highest mean |SHAP| value? Does that align with your intuition from the Workshop 1–3 visualizations?
3. If you were presenting these results to a county planning board, which SHAP output would you show first — the global bar plot or the waterfall for a specific county? Why?
4. What additional spatial features might improve the model's accuracy?


*Write your answers here.*

1.
2.
3.
4.


## Key Takeaways

| Concept | What you did | Why it matters |
|---------|-------------|----------------|
| **Moran's I** | Tested spatial autocorrelation of access rates | Confirms geographic clustering — a violation of i.i.d. assumption |
| **Spatial lags** | Engineered neighbor-average features | Let the model "see" what surrounds each county |
| **Random Forest + LOO-CV** | Trained on 20, predicted 1, repeated 21× | Honest accuracy estimate for small spatial datasets |
| **SHAP** | Decomposed each prediction into feature contributions | Turned black-box model into a policy-ready argument |
| **Prediction map** | Mapped predicted class and probability | Verified model makes geographic sense |

---
### Extension Challenge
Spatial CV is more rigorous than LOO-CV for spatial data. Implement **spatial k-fold cross-validation** using `sklearn` + a manual spatial partition (e.g., north vs. south NJ). Does the accuracy estimate change?

---
### Learning Objectives Check-in
- [ ] I can explain what Moran's I measures and interpret its significance
- [ ] I can build spatial lag features using a queen contiguity weight matrix
- [ ] I can train a Random Forest and evaluate it with LOO-CV
- [ ] I can interpret SHAP global and local explanations
- [ ] I can map model predictions and identify misclassified counties

**Next workshop (Nov 17): GIS Workshop 5 — Accessibility & Network Analysis**
We add a new spatial layer: how far can NJ residents travel (by road) to reach a library or broadband access point?
